# Physical multilevel MOT trajectory explorer

This notebook drives the Section-12 24-state population-rate solver. Every RK4 stage recomputes the magnetic field, polarization projection, detunings, Rabi frequencies, rate matrix, steady populations, and net stimulated force. Recoil diffusion is not included.

In [ ]:
%matplotlib widget
from dataclasses import asdict, replace
from datetime import datetime
from pathlib import Path
from time import perf_counter

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pmot.magnetic_fields import default_anti_helmholtz_config
from pmot.mot_multilevel import (
    RateEquationAtomState, RateEquationTrajectoryConfig,
    build_multilevel_mot_beams, build_rate_equation_model,
    default_multilevel_mot_config, multilevel_mot_paths,
    plot_time_diagnostics, plot_trajectory_3d, save_trajectory,
    simulate_rate_equation_trajectory, trajectory_summary, trajectory_table,
)

MODEL = build_rate_equation_model()
OUTPUT = multilevel_mot_paths()['trajectories'] / 'notebook_trajectory_explorer'

In [ ]:
x0 = widgets.FloatText(description='x0 [mm]', value=8.0)
y0 = widgets.FloatText(description='y0 [mm]', value=1.0)
z0 = widgets.FloatText(description='z0 [mm]', value=0.0)
vx0 = widgets.FloatText(description='vx0 [m/s]', value=-3.0)
vy0 = widgets.FloatText(description='vy0 [m/s]', value=0.1)
vz0 = widgets.FloatText(description='vz0 [m/s]', value=0.0)
duration = widgets.FloatSlider(description='T [ms]', min=0.1, max=50.0, step=0.1, value=5.0, continuous_update=False)
dt = widgets.FloatSlider(description='dt [us]', min=1.0, max=20.0, step=1.0, value=5.0, continuous_update=False)
gradient = widgets.FloatSlider(description='dBz/dz [G/cm]', min=1, max=30, step=.5, value=10, continuous_update=False)
detuning = widgets.FloatSlider(description='cool Δ [MHz]', min=-40.0, max=-1.0, step=0.5, value=-15.0, continuous_update=False)
cooling_power = widgets.FloatSlider(description='cool P [mW]', min=0.1, max=60.0, step=0.1, value=27.0, continuous_update=False)
repump_power = widgets.FloatSlider(description='repump P [mW]', min=0.01, max=2.0, step=0.01, value=0.1, continuous_update=False)
gravity = widgets.Checkbox(description='gravity', value=True, indent=False)
save = widgets.Checkbox(description='save CSV/NPZ/figures', value=False, indent=False)
run = widgets.Button(description='Run trajectory', button_style='primary')
out = widgets.Output()

def run_trajectory(_=None):
    with out:
        out.clear_output(wait=True)
        plt.close('all')
        config = replace(
            default_multilevel_mot_config(),
            cooling_detuning_rad_per_s=2*np.pi*detuning.value*1e6,
            cooling_power_w_per_beam=cooling_power.value*1e-3,
            repump_power_w_per_beam=repump_power.value*1e-3,
            include_gravity=gravity.value,
        )
        coil = default_anti_helmholtz_config(target_gradient_g_per_cm=gradient.value)
        beams = build_multilevel_mot_beams(config=config)
        initial = RateEquationAtomState(
            tuple(1e-3*np.asarray([x0.value, y0.value, z0.value])),
            (vx0.value, vy0.value, vz0.value),
        )
        numerical = RateEquationTrajectoryConfig(time_step_s=dt.value*1e-6)
        print(f'Running {duration.value:g} ms at {dt.value:g} µs ({int(np.ceil(1000*duration.value/dt.value)):,} RK4 steps) ...')
        started = perf_counter()
        record = simulate_rate_equation_trajectory(
            initial, duration.value*1e-3, coil, beams=beams, model=MODEL,
            config=config, trajectory_config=numerical,
        )
        summary = trajectory_summary(record)
        print({**summary, 'wall_time_s': perf_counter()-started})
        plot_trajectory_3d(record, beams)
        plot_time_diagnostics(record, MODEL, beams)
        display(trajectory_table(record).iloc[::max(1, len(record.times_s)//20)])
        if save.value:
            stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            stem = OUTPUT / f'trajectory_{stamp}'
            files = save_trajectory(record, MODEL, beams, stem, metadata={'config': asdict(config), 'coil_config': asdict(coil)})
            plot_trajectory_3d(record, beams, stem.with_name(stem.name+'_3d.png'))
            plot_time_diagnostics(record, MODEL, beams, stem.with_name(stem.name+'_diagnostics.png'))
            print('Saved:'); [print(' ', path) for path in files]
        plt.show()

run.on_click(run_trajectory)
display(widgets.VBox([
    widgets.HBox([x0, y0, z0]), widgets.HBox([vx0, vy0, vz0]),
    widgets.HBox([duration, dt, gradient, gravity]),
    widgets.HBox([detuning, cooling_power, repump_power]),
    widgets.HBox([save, run]), out,
]))